# Block 18 — LAB: Time Series — ARIMA
### Advanced Machine Learning — M&T Bank

Loads `macro_vix_monthly.csv` (unchanged since Block 17). Same chronological split. No new CSV comes out of this
lab.

Block 17 read the PACF: a sharp cutoff after lag 1, suggesting an AR(1)-shaped process.

**Part 1 — Fit the model the PACF suggested**, and watch a static multi-step forecast fail in a specific,
diagnosable way.

**Part 2 — Fix the evaluation**: walk-forward, one step at a time, the way a forecast would actually be used.

**Part 3 — Check nearby orders**: does anything beat the order the diagnostic plot already suggested?

**Part 4 — Residual diagnostics**: is anything still left on the table?

Look for `# TODO` — that's where your code goes. Each task has a hint; ask if you get stuck.


## Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)

NAVY = "#251E4E"
PINK = "#FF1675"
GRAY = "#6b7280"
ORANGE = "#FF7B01"

# Data source: https://raw.githubusercontent.com/mithun-rk/mt-advml-course/main/Data/macro_vix_monthly.csv
vix = pd.read_csv("https://raw.githubusercontent.com/mithun-rk/mt-advml-course/main/Data/macro_vix_monthly.csv", sep=";", parse_dates=["date"]).set_index("date")
vix.index.freq = "MS"
series = vix["vix_close"]

split_idx = int(len(vix) * 0.8)
train, test = series.iloc[:split_idx], series.iloc[split_idx:]
print(f"Train: {len(train)} months. Test: {len(test)} months (same chronological split as Block 17).")


## Part 1 — Fit ARIMA(1,0,0), the Order the PACF Suggested

`ARIMA(p, d, q)`: `p` = autoregressive lags, `d` = differencing order, `q` = moving-average lags. Block 17 found
the series stationary (`d=0`) and a PACF that cuts off sharply after lag 1 (`p=1`).

### TODO 1.1 — Fit it

`ARIMA(train, order=(1, 0, 0)).fit()`


In [ ]:
# TODO: model = ARIMA(train, order=(1, 0, 0)).fit()
# TODO: print(model.summary().tables[1])
# TODO: print(f"AIC: {model.aic:.1f}")


### TODO 1.2 — Forecast the whole test period in one shot

`model.forecast(steps=len(test))` — the most obvious way to use a fitted model. Plot it against the actual test
values, then compute MAE and RMSE.


In [ ]:
# TODO: static_fc = model.forecast(steps=len(test))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train.index[-36:], train.values[-36:], label="Train (last 3 yrs)", color=NAVY, linewidth=1)
ax.plot(test.index, test.values, label="Actual (test)", color=NAVY, linewidth=1.2)
# TODO: ax.plot(test.index, static_fc.values, label="Static ARIMA(1,0,0) forecast", color=PINK, linewidth=1.5)
ax.legend()
plt.tight_layout()
plt.show()

# TODO: static_mae = np.mean(np.abs(test.values - static_fc.values))
# TODO: static_rmse = np.sqrt(np.mean((test.values - static_fc.values) ** 2))
# TODO: print both. Compare to Block 17's naive baseline (MAE 3.192 / RMSE 5.600) — better or worse?


**Question to answer:** look at the chart. What does the pink forecast line do a few months in, and why? (Hint:
what does an AR(1) process do when it's forecast many steps ahead with no new real observations feeding in?)


## Part 2 — Walk-Forward Evaluation

A more realistic test: at each month in the test period, forecast only *one step ahead*, using every real
observation up through the previous month (an expanding window).

### TODO 2.1 — Write the walk-forward loop

For each `t` from `split_idx` to `len(series)`: fit `ARIMA(history, order=order)` on everything seen so far,
forecast 1 step, record it, then append the *real* value at `t` to `history` before moving on.


In [ ]:
def walk_forward(order, series, split_idx):
    history = list(series.iloc[:split_idx])
    preds = []
    for t in range(split_idx, len(series)):
        # TODO: m = ARIMA(history, order=order).fit()
        # TODO: preds.append(m.forecast(steps=1)[0])
        # TODO: history.append(series.iloc[t])
        pass
    return np.array(preds)

# TODO: wf_preds = walk_forward((1, 0, 0), series, split_idx)
# TODO: wf_mae = np.mean(np.abs(test.values - wf_preds))
# TODO: wf_rmse = np.sqrt(np.mean((test.values - wf_preds) ** 2))
# TODO: print both


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(test.index, test.values, label="Actual", color=NAVY, linewidth=1.2)
# TODO: ax.plot(test.index, wf_preds, label="Walk-forward ARIMA(1,0,0)", color=PINK, linewidth=1.2)
ax.legend()
plt.tight_layout()
plt.show()


**Question to answer:** how does the walk-forward MAE/RMSE compare to Part 1's static forecast, and to Block
17's naive and lag-1-regression baselines? Which of Block 17's two baselines does ARIMA(1,0,0) end up closest to,
and why might that be?


## Part 3 — Does a Different Order Do Better?

### TODO 3.1 — Try a small grid: `(2,0,0)`, `(1,0,1)`, `(2,0,1)`

Run `walk_forward` for each and compare MAE/RMSE against `(1,0,0)`'s result from Part 2.


In [ ]:
order_results = [{"order": (1, 0, 0), "MAE": round(wf_mae, 3), "RMSE": round(wf_rmse, 3)}]
for order in [(2, 0, 0), (1, 0, 1), (2, 0, 1)]:
    # TODO: preds = walk_forward(order, series, split_idx)
    # TODO: mae = ...; rmse = ...
    # TODO: order_results.append({"order": order, "MAE": round(mae, 3), "RMSE": round(rmse, 3)})
    pass

pd.DataFrame(order_results)


**Question to answer:** did any of the more complex orders beat `(1,0,0)`? What does that say about whether the
PACF chart from Block 17 was actually useful for choosing a model order, versus just a nice-looking diagnostic?


## Part 4 — Residual Diagnostics: Is Anything Left on the Table?

If ARIMA(1,0,0) fully captured the structure, its residuals should look like white noise.

### TODO 4.1 — Run the Ljung-Box test on `model.resid` (the Part-1 fitted model)

`acorr_ljungbox(resid, lags=[10], return_df=True)`


In [ ]:
# TODO: resid = model.resid
# TODO: lb = acorr_ljungbox(resid, lags=[10], return_df=True)
# TODO: print(lb)
# TODO: print(f"Residual mean: {resid.mean():.3f}, residual std: {resid.std():.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
# TODO: ax.plot(resid.index, resid.values, color=GRAY, linewidth=0.8)
ax.axhline(0, color=NAVY, linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()


**Questions to answer:**
1. Is the Ljung-Box p-value above or below 0.05? What does that say about whether any autocorrelation is left in
   the residuals?
2. VIX is known for *volatility clustering* — calm periods and turbulent periods each tend to persist. Is that a
   pattern in the residuals' *level* or in their *variance*? Would a plain ARIMA model, which is linear, be
   expected to capture that kind of pattern?


## Wrap-Up

Write 2-3 sentences: did the static forecast or the walk-forward forecast better reflect how this model would
actually be used in production, and why? Did the PACF chart from Block 17 turn out to be a reliable guide to
choosing the model order? What did the residual diagnostics suggest ARIMA still can't capture about this series?
